In [1]:
import shutup; shutup.please()
import pandas as pd
import numpy as np
import os

In [3]:
#Load Load Prices
def load_load_values(files: list):
    load_df = pd.DataFrame()

    for file in files:
        df = pd.read_csv(f"Entso_e\Load\{file}", sep=",")
        df.drop('Area', axis=1, inplace=True)
        df["MTU (CET/CEST)"] = df["MTU (CET/CEST)"].apply(lambda x: x[:16])
        df["MTU (CET/CEST)"] = pd.to_datetime(df["MTU (CET/CEST)"],dayfirst=True)
        df.index = df["MTU (CET/CEST)"]
        # get that column that has actual in it
        df = df.loc[:,df.columns.str.contains('forecast', case=False)]
        # rename that one column to just demand
        df = df.rename(columns={df.columns[0]:'load_forecast'})
        df = df.replace('n/e', np.nan)
        df = df.replace("N/A", np.nan)
        df = df.replace("", np.nan)
        df = df.replace(" ", np.nan)
        df.index.name = "date"
        load_df = pd.concat([load_df, df], axis=0)

    return load_df



<string>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\L'
<string>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\L'
C:\Users\local_user\AppData\Local\Temp\ipykernel_22732\1165384407.py:6: SyntaxWarning: invalid escape sequence '\{'
  df = pd.read_csv(f"Entso_e\Load\{file}", sep=",")
C:\Users\local_user\AppData\Local\Temp\ipykernel_22732\1165384407.py:6: SyntaxWarning: invalid escape sequence '\L'
  df = pd.read_csv(f"Entso_e\Load\{file}", sep=",")


In [4]:
#Load Da Prices
def load_da_prices(files: list):
    da_df = pd.DataFrame()

    for file in files:
        df = pd.read_csv(f"Entso_e\DA_price\{file}", sep=",")
        df.drop('Area', axis=1, inplace=True)
        df.drop('Sequence', axis=1, inplace=True)
        if "Intraday Period (CET/CEST)" in df.columns:
            df.drop('Intraday Period (CET/CEST)', axis=1, inplace=True)
        if "Intraday Price (EUR/MWh)" in df.columns:
            df.drop("Intraday Price (EUR/MWh)", axis=1, inplace=True)
        
        df["MTU (CET/CEST)"] = df["MTU (CET/CEST)"].apply(lambda x: x[:16])
        df["MTU (CET/CEST)"] = pd.to_datetime(df["MTU (CET/CEST)"],dayfirst=True)
        df.set_index("MTU (CET/CEST)", inplace=True)
        df = df.rename(columns={df.columns[0]:'hupx_prices'})
        df = df.replace('n/e', np.nan)
        df = df.replace("N/A", np.nan)
        df = df.replace("", np.nan)
        df = df.replace(" ", np.nan)
        df.index.name = "date"
        da_df = pd.concat([da_df, df], axis=0)
        
    return da_df


<string>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\D'
<string>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\{'
<>:6: SyntaxWarning: invalid escape sequence '\D'
C:\Users\local_user\AppData\Local\Temp\ipykernel_22732\3311860771.py:6: SyntaxWarning: invalid escape sequence '\{'
  df = pd.read_csv(f"Entso_e\DA_price\{file}", sep=",")
C:\Users\local_user\AppData\Local\Temp\ipykernel_22732\3311860771.py:6: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv(f"Entso_e\DA_price\{file}", sep=",")


In [22]:
#Time Features
def time_features(df: pd.DataFrame):
    df = df.copy()
    df["date"] = df.index
    df["dayofweek"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["dayofyear"] = df["date"].dt.dayofyear
    df["dayofmonth"] = df["date"].dt.day
    df["QH_slot"] = df.index.hour*4 + df.index.minute // 15
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["sin_qh"] = np.sin(2*np.pi*df["QH_slot"] /96)
    df["cos_qh"] = np.cos(2*np.pi*df["QH_slot"] /96)

    df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dayofwee_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

    # Month (period = 12)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

    # Day of year (period = 365.25 to smooth leap years)
    df["dayofyear_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["dayofyear_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)

    # Day of month (period = 31)
    df["dayofmonth_sin"] = np.sin(2 * np.pi * (df["dayofmonth"] - 1) / 31)
    df["dayofmonth_cos"] = np.cos(2 * np.pi * (df["dayofmonth"] - 1) / 31)


    lags = [96, 96*2, 96*3, 96*7]  # 1d, 2d, 3d, 7d

    for lag in lags:
        df[f"lag_{lag}"] = df["hupx_prices"].shift(lag)

    df[f"roll_mean_24"] = df["hupx_prices"].shift(1).rolling(window=96, min_periods=1).mean()
    df[f"roll_std_24"]  = df["hupx_prices"].shift(1).rolling(window=96, min_periods=1).std()
    df.drop("dayofweek", inplace=True, axis=1)
    df.drop("month", inplace=True, axis=1)
    df.drop("dayofyear", inplace=True, axis=1)
    df.drop("dayofmonth", inplace=True, axis=1)
    df.drop("date", inplace=True, axis=1)
    df.dropna(inplace=True)
    return df
    


In [23]:
#Combine the Two
load_path = os.listdir(os.path.join("Entso_e\Load"))
load_df = load_load_values(load_path)
da_path = os.listdir(os.path.join("Entso_e\DA_price"))
da_df = load_da_prices(da_path)

#Combine the Two
Combined_df = da_df.join(load_df)

#Get the data where the 15 min period starts
Combined_df = Combined_df.loc["2025-10-1":]
Combined_df = time_features(Combined_df)
Combined_df

<>:2: SyntaxWarning: invalid escape sequence '\L'
<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\L'
<>:4: SyntaxWarning: invalid escape sequence '\D'
C:\Users\local_user\AppData\Local\Temp\ipykernel_22732\3238029669.py:2: SyntaxWarning: invalid escape sequence '\L'
  load_path = os.listdir(os.path.join("Entso_e\Load"))
C:\Users\local_user\AppData\Local\Temp\ipykernel_22732\3238029669.py:4: SyntaxWarning: invalid escape sequence '\D'
  da_path = os.listdir(os.path.join("Entso_e\DA_price"))


,hupx_prices,load_forecast,QH_slot,is_weekend,sin_qh,cos_qh,dayofweek_sin,dayofwee_cos,month_sin,month_cos,dayofyear_sin,dayofyear_cos,dayofmonth_sin,dayofmonth_cos,lag_96,lag_192,lag_288,lag_672,roll_mean_24,roll_std_24
date,,,,,,,,,,,,,,,,,,,,
2025-10-08 00:00:00,101.74,4657.52,0,0,0.000000,1.000000,0.974928,-0.222521,-1.0,-1.836970e-16,-0.992629,0.121193,0.988468,0.151428,105.51,109.59,83.25,108.98,138.180417,56.074910
2025-10-08 00:15:00,91.40,4559.25,1,0,0.065403,0.997859,0.974928,-0.222521,-1.0,-1.836970e-16,-0.992629,0.121193,0.988468,0.151428,102.60,102.31,81.72,97.08,138.141146,56.099346
2025-10-08 00:30:00,91.47,4472.02,2,0,0.130526,0.991445,0.974928,-0.222521,-1.0,-1.836970e-16,-0.992629,0.121193,0.988468,0.151428,100.26,96.73,55.68,90.97,138.024479,56.185616
2025-10-08 00:45:00,89.20,4395.84,3,0,0.195090,0.980785,0.974928,-0.222521,-1.0,-1.836970e-16,-0.992629,0.121193,0.988468,0.151428,96.12,87.97,13.00,90.60,137.932917,56.254926
2025-10-08 01:00:00,91.85,4329.85,4,0,0.258819,0.965926,0.974928,-0.222521,-1.0,-1.836970e-16,-0.992629,0.121193,0.988468,0.151428,100.00,83.00,27.86,91.15,137.860833,56.313471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-24 22:45:00,106.19,5458.86,91,0,-0.321439,0.946930,0.781831,0.623490,0.5,8.660254e-01,0.811160,0.584824,-0.998717,-0.050649,94.93,59.27,71.37,97.65,114.829792,26.438392
2026-02-24 23:00:00,103.46,5399.73,92,0,-0.258819,0.965926,0.781831,0.623490,0.5,8.660254e-01,0.811160,0.584824,-0.998717,-0.050649,95.44,76.23,85.69,101.44,114.947083,26.374078
2026-02-24 23:15:00,101.93,5331.98,93,0,-0.195090,0.980785,0.781831,0.623490,0.5,8.660254e-01,0.811160,0.584824,-0.998717,-0.050649,92.39,60.19,78.83,97.23,115.030625,26.324292


In [25]:
#Save
Combined_df.to_csv(os.path.join("processed_data", "processed_data.csv"))